# 🦙 QLoRA Fine-Tuning: Llama 3.2-3B × Dolly-15K

**Yöntem / Method:** QLoRA (4-bit Quantization + LoRA Adapters)  
**Model:** `unsloth/Llama-3.2-3B-Instruct`  
**Dataset:** `databricks/databricks-dolly-15k` — 500 örnek subset  
**Tracking:** MLflow experiment tracking  
**Hedef / Goal:** Genel instruction-following yeteneğini artırmak

---

## ⚙️ Başlamadan Önce / Before Starting
1. **Runtime → Change runtime type → T4 GPU** seç
2. Sol menü 🔑 **Secrets** bölümüne `HF_TOKEN` ekle → [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)

---

## 📋 Notebook Akışı / Flow
| Bölüm | İçerik |
|---|---|
| 1 | Kurulum |
| 2 | GPU & HuggingFace bağlantısı |
| 3 | Model yükleme (4-bit QLoRA) |
| 4 | LoRA adaptörleri ekleme |
| 5 | Dolly-15K veri seti hazırlama |
| 6 | MLflow experiment tracking |
| 7 | Fine-tuning eğitimi |
| 8 | Model testi |
| 9 | HuggingFace'e yükleme |

## 📦 Bölüm 1: Kurulum / Installation

- `unsloth` → QLoRA eğitimini 2x hızlandırır, %70 daha az RAM kullanır
- `trl` → SFTTrainer (Supervised Fine-Tuning döngüsü)
- `peft` → LoRA/QLoRA adaptör sistemi
- `mlflow` → Deney takibi / Experiment tracking

> ⏱️ İlk çalıştırmada ~3-5 dakika sürer / Takes ~3-5 min on first run

In [1]:
# Gerekli kütüphaneleri kur
# Install required libraries
!pip install unsloth -q
!pip install "transformers>=4.43.0" trl peft accelerate bitsandbytes datasets mlflow -q

print('✅ Kurulum tamamlandı / Installation complete')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224

## 🖥️ Bölüm 2: GPU Kontrolü & HuggingFace Girişi
/ GPU Check & HuggingFace Login

- T4 GPU seçiliyse **14.5 GB** VRAM görürsün
- HF token olmadan Llama modeli indirilemez (gated model)
- Token'ı Colab **Secrets** bölümüne ekle, koda yazma → güvenli değil

In [2]:
import torch
from google.colab import userdata
from huggingface_hub import login

# GPU kontrolü / GPU check
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK — T4 seç!"}')
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# HuggingFace girişi / HuggingFace login
# Token'ı Secrets'tan oku — direkt yazmak güvensiz
# Read token from Secrets — writing it directly is insecure
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print('✅ HuggingFace bağlantısı başarılı / Login successful')
except Exception as e:
    print(f'⚠️ Token bulunamadı: {e}')
    print('Çözüm: Sol menü 🔑 Secrets > HF_TOKEN ekle')

GPU: Tesla T4
VRAM: 15.6 GB
✅ HuggingFace bağlantısı başarılı / Login successful


## 🔧 Bölüm 3: Model Yükleme — 4-bit QLoRA
/ Model Loading — 4-bit QLoRA

**QLoRA nasıl çalışır? / How QLoRA works:**
1. Model ağırlıkları **4-bit**'e sıkıştırılır → RAM ~6-8 GB'a düşer (normalde ~28 GB)
2. Hesaplama sırasında **bfloat16**'ya dönüştürülür (dequantization)
3. Orijinal ağırlıklar **dondurulur** — sadece LoRA adaptörleri eğitilir

| Yöntem | VRAM (3B model) |
|---|---|
| Full fine-tuning | ~24 GB |
| QLoRA (4-bit) | ~4-5 GB ✅ |

> `load_in_4bit=True` → Unsloth her şeyi otomatik halleder

In [4]:
!pip install modelscope -q

import os
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'  # HF down olursa ModelScope'tan indir
                                              # Fallback to ModelScope if HF is down

from unsloth import FastLanguageModel

# Hiperparametreler / Hyperparameters
MAX_SEQ_LENGTH = 1024  # Token uzunluğu — Dolly için 1024 yeterli
                        # Token length — 1024 is enough for Dolly
MODEL_NAME = 'unsloth/Llama-3.2-3B-Instruct'

print('Model yükleniyor... (~2-3 dk / min)')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # Otomatik algıla / Auto-detect
    load_in_4bit=True,   # QLoRA — 4-bit quantization
    token=hf_token,
)

total = sum(p.numel() for p in model.parameters())
print(f'✅ Model yüklendi: {MODEL_NAME}')
print(f'Parametre sayısı / Parameters: {total/1e9:.2f}B')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 84.9 MB/s eta 0:00:00


/tmp/ipykernel_897/3726399862.py:6: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth Zoo will now patch everything to make training faster!
Model yükleniyor... (~2-3 dk / min)


2026-03-10 18:50:25,702 - modelscope - INFO - Got 9 files, start to download ...


Processing 9 items:   0%|          | 0.00/9.00 [00:00<?, ?it/s]

2026-03-10 18:54:39,984 - modelscope - INFO - Download model 'unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit' successfully.
2026-03-10 18:54:39,985 - modelscope - INFO - Creating symbolic link [/root/.cache/modelscope/hub/models/unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit].


==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load /root/.cache/modelscope/hub/models/unsloth/llama-3___2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


✅ Model yüklendi: unsloth/Llama-3.2-3B-Instruct
Parametre sayısı / Parameters: 1.84B


## 🎯 Bölüm 4: LoRA Adaptörleri / LoRA Adapters

**Ne oluyor? / What happens here?**
- Orijinal 3B ağırlık **dondurulur** → değişmez
- 7 katmana küçük **adaptör matrisleri** eklenir
- Sadece bu ~24M parametre eğitilir (%0.75)

**Neden bu 7 katman? / Why these 7 layers?**
- `q/k/v_proj` → Attention: modelin neye odaklandığı
- `o_proj` → Attention çıktısı
- `gate/up/down_proj` → FFN: bilgiyi işleme katmanları

**`r=16` ne demek?** Adaptör matrisinin boyutu. Yüksek = daha fazla kapasite, daha fazla RAM. 16 iyi denge noktası.

In [5]:
# LoRA konfigürasyonu — donmuş model üzerine adaptör ekle
# LoRA config — add adapters on top of frozen model
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                        # Adaptör rank / Adapter rank
    lora_alpha=16,               # Öğrenme ölçeği (r ile aynı = iyi başlangıç)
                                  # Learning scale (same as r = good start)
    lora_dropout=0,              # 0 = Unsloth tam hız optimize eder
                                  # 0 = Unsloth optimizes at full speed
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',  # Attention katmanları
        'gate_proj', 'up_proj', 'down_proj',       # FFN katmanları
    ],
    bias='none',
    use_gradient_checkpointing='unsloth',  # RAM tasarrufu / RAM saving
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Eğitilecek / Trainable: {trainable/1e6:.1f}M  ({100*trainable/total:.2f}%)')
print(f'Donmuş / Frozen:        {(total-trainable)/1e9:.2f}B')
print('← Küçük oran = QLoRA\'nın gücü! / Small ratio = power of QLoRA!')

Unsloth 2026.3.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Eğitilecek / Trainable: 24.3M  (1.30%)
Donmuş / Frozen:        1.84B
← Küçük oran = QLoRA'nın gücü! / Small ratio = power of QLoRA!


## 📚 Bölüm 5: Veri Seti — Databricks Dolly-15K
/ Dataset — Databricks Dolly-15K

**Neden Dolly-15K? / Why Dolly-15K?**
- Databricks tarafından insan tarafından yazılmış 15.000 instruction-response çifti
- 8 kategori: QA, özetleme, sınıflandırma, yaratıcı yazı, bilgi çıkarımı...
- Temiz, lisanslı (CC BY-SA 3.0), akademik referans


**500 örnek neden yeterli?** LoRA ile az veriyle iyi sonuç alınır. Overfitting'i önlemek için 500 denge noktası.
**Llama-3 Chat Formatı:**
```
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Sen yardımcı bir asistansın.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Soru<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
Cevap<|eot_id|>
```

In [6]:
from datasets import load_dataset

# Dolly-15K'yı yükle, 500 örnek al
# Load Dolly-15K, take 500 examples
print('Dolly-15K yükleniyor... / Loading Dolly-15K...')
raw = load_dataset('databricks/databricks-dolly-15k', split='train')
raw = raw.shuffle(seed=42).select(range(500))
print(f'Yüklendi / Loaded: {len(raw)} örnek/examples')
print(f'Kategoriler / Categories: {set(raw["category"])}')

Dolly-15K yükleniyor... / Loading Dolly-15K...


README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Yüklendi / Loaded: 500 örnek/examples
Kategoriler / Categories: {'summarization', 'information_extraction', 'open_qa', 'closed_qa', 'brainstorming', 'general_qa', 'classification', 'creative_writing'}


In [8]:
# Llama-3 chat formatı şablonu
# Llama-3 chat format template
LLAMA3_PROMPT = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a helpful, accurate, and concise AI assistant.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{instruction}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
{response}<|eot_id|>"""

def format_dolly(example):
    """
    Dolly formatını Llama-3 chat formatına çevirir.
    Converts Dolly format to Llama-3 chat format.

    Dolly alanları / Dolly fields:
      instruction → kullanıcı sorusu / user question
      context     → ek bağlam (opsiyonel) / extra context (optional)
      output      → beklenen cevap / expected answer
    """
    instruction = example['instruction']
    context     = example.get('context', '').strip()
    response = example['response']

    # Context varsa instruction'a ekle
    # If context exists, append to instruction
    if context:
        instruction = f'{instruction}\n\nContext:\n{context}'

    return {
        'text': LLAMA3_PROMPT.format(
            instruction=instruction,
            response=response
        ) + tokenizer.eos_token  # Modele "cevap bitti" sinyali / Signal end of response
    }

# Veriyi formatla ve böl
# Format and split data
dataset     = raw.map(format_dolly, remove_columns=raw.column_names)
split       = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
eval_dataset  = split['test']

print(f'Eğitim / Train:      {len(train_dataset)} örnek')
print(f'Validasyon / Val:    {len(eval_dataset)} örnek')
print(f'\n--- Örnek formatlı girdi / Sample formatted input ---')
print(train_dataset[0]['text'][:400] + '...')

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Eğitim / Train:      450 örnek
Validasyon / Val:    50 örnek

--- Örnek formatlı girdi / Sample formatted input ---
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a helpful, accurate, and concise AI assistant.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
What are the top largest economies in the world?<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
The United States
China
Japan
Germany<|eot_id|><|eot_id|>...


## 📊 Bölüm 6: MLflow — Deney Takibi / Experiment Tracking

**MLflow ne yapar? / What does MLflow do?**
- Hangi hiperparametre ile hangi sonucu aldığını kaydeder
- Farklı deneyleri karşılaştırmana izin verir

```
Experiment (grup)  →  llama3-dolly-finetuning
  Run (tek deneme) →  qlora-r16-500examples
    Params         →  model, r, lr, dataset...
    Metrics        →  train_loss, val_loss, duration
    Artifacts      →  config.json
```

In [9]:
import mlflow, json

# Deney grubunu oluştur / Create experiment group
mlflow.set_experiment('llama3-dolly-finetuning')

# Run başlat / Start run
run = mlflow.start_run(run_name='qlora-r16-500ex-dolly')

# Hiperparametreleri logla / Log hyperparameters
mlflow.log_params({
    'model':           'Llama-3.2-3B-Instruct',
    'dataset':         'databricks/dolly-15k',
    'train_examples':  len(train_dataset),
    'quantization':    '4-bit QLoRA',
    'lora_r':          16,
    'lora_alpha':      16,
    'max_seq_length':  MAX_SEQ_LENGTH,
    'learning_rate':   '2e-4',
})

print(f'✅ MLflow run başlatıldı / MLflow run started')
print(f'Run ID: {run.info.run_id}')

2026/03/10 19:18:09 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/10 19:18:09 INFO mlflow.store.db.utils: Updating database tables
2026/03/10 19:18:10 INFO mlflow.tracking.fluent: Experiment with name 'llama3-dolly-finetuning' does not exist. Creating a new experiment.


✅ MLflow run başlatıldı / MLflow run started
Run ID: e91e1a6010ba40edada20583410201bd


## 🏋️ Bölüm 7: Fine-Tuning Eğitimi / Training

**SFTTrainer:** HuggingFace TRL'den Supervised Fine-Tuning trainer.  
Eğitim döngüsünü yönetir: forward pass → loss → backward → güncelle.

**Kritik hiperparametreler / Key hyperparameters:**

| Parametre | Değer | Neden |
|---|---|---|
| `per_device_train_batch_size` | 2 | T4 için güvenli |
| `gradient_accumulation_steps` | 4 | Etkili batch = 8 |
| `max_steps` | 120 | 500 örnek için yeterli |
| `learning_rate` | 2e-4 | Fine-tuning standardı |
| `warmup_steps` | 10 | İlk adımlarda stabilite |

> ⏱️ T4'te ~20 dakika / ~20 minutes on T4  


In [10]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,

    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,   # Etkili batch = 8 / Effective batch = 8
        warmup_steps=10,                 # İlk 10 adımda lr yavaş artar
        max_steps=120,                   # 500 örnek için yeterli / Enough for 500 examples
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,                # Her 10 adımda loss yazdır / Print loss every 10 steps
        optim='adamw_8bit',              # 8-bit optimizer — daha az RAM / less RAM
        weight_decay=0.01,
        lr_scheduler_type='cosine',      # Cosine decay — linear'dan daha smooth
        seed=42,
        output_dir='./qlora_dolly_output',
        eval_strategy='steps',
        eval_steps=30,
        save_strategy='steps',
        save_steps=60,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
    ),
)

print('✅ Trainer hazır / Trainer ready')

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/450 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

✅ Trainer hazır / Trainer ready


In [11]:
import time
torch.cuda.empty_cache()  # GPU hafızasını temizle / Clear GPU memory

print('🚀 Eğitim başlıyor / Training starting...')
print('Loss her 10 adımda görünür — ikisi de azalıyorsa iyi!')
print('Both train & val loss should decrease — that means no overfitting!')
print('-' * 60)

t0           = time.time()
trainer_stats = trainer.train()
elapsed      = time.time() - t0

print(f'\n✅ Eğitim tamamlandı / Training complete!')
print(f'Süre / Duration:   {elapsed/60:.1f} dakika/min')
print(f'Son loss / Final:  {trainer_stats.training_loss:.4f}')

# MLflow'a sonuçları logla / Log results to MLflow
mlflow.log_metrics({
    'final_train_loss':      trainer_stats.training_loss,
    'training_minutes':      round(elapsed / 60, 2),
    'total_steps':           trainer_stats.global_step,
})
print('📊 MLflow\'a loglandı / Logged to MLflow')

🚀 Eğitim başlıyor / Training starting...
Loss her 10 adımda görünür — ikisi de azalıyorsa iyi!
Both train & val loss should decrease — that means no overfitting!
------------------------------------------------------------


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 450 | Num Epochs = 3 | Total steps = 120
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss,Validation Loss
30,1.872863,1.772028
60,1.779140,1.583299
90,1.744904,1.576932
120,1.491095,1.574595


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_co


✅ Eğitim tamamlandı / Training complete!
Süre / Duration:   6.2 dakika/min
Son loss / Final:  1.8792
📊 MLflow'a loglandı / Logged to MLflow


## 🧪 Bölüm 8: Model Testi / Model Testing

Eğitim bitti. Modele birkaç farklı kategoriden soru sorup kaliteyi değerlendir.

**İyi sonuç kriterleri / Good result criteria:**
- Soruyla alakalı cevap üretiyor mu?
- Cevap makul uzunlukta mı?
- Saçma/tekrarlayan bir şey yazıyor mu?

> `temperature=0.7` → biraz yaratıcılık. `0.1` yapsan daha deterministik olur.

In [12]:
# Inference moduna geç — eğitimden daha hızlı
# Switch to inference mode — faster than training mode
FastLanguageModel.for_inference(model)

def ask(question, max_new_tokens=256):
    """Modele soru sor ve cevabı döndür / Ask model a question and return answer"""
    prompt = LLAMA3_PROMPT.format(instruction=question, response='')
    inputs = tokenizer([prompt], return_tensors='pt').to('cuda')

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Sadece yeni tokenleri al (prompt hariç)
    # Get only new tokens (excluding prompt)
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# Dolly'nin 8 kategorisinden test soruları
# Test questions from Dolly's 8 categories
tests = [
    ('open_qa',      'What causes the Northern Lights?'),
    ('classification','Is the following sentence positive or negative? "I absolutely loved the movie!"'),
    ('summarization', 'Summarize this in one sentence: The French Revolution began in 1789 and ended in 1799. It led to the rise of Napoleon Bonaparte and fundamentally changed France\'s political structure.'),
    ('brainstorming', 'Give me 3 creative names for a coffee shop.'),
]

print('🧪 Model Testi / Model Testing')
print('=' * 60)
for category, q in tests:
    print(f'\n[{category}]')
    print(f'Q: {q}')
    print(f'A: {ask(q)}')
    print()

🧪 Model Testi / Model Testing

[open_qa]
Q: What causes the Northern Lights?
A: Context
The Northern Lights, also known as the aurora borealis, are a natural light display that occurs when charged particles from the sun interact with the Earth's magnetic field and atmosphere. The aurora is most commonly seen in the northernmost parts of the world, such as Alaska, Canada, Sweden, Norway, and Russia. The northern lights are a natural wonder that has captivated people for centuries, and they are considered a unique and breathtaking sight.

The northern lights are caused by the interaction between the Earth's magnetic field and the solar wind, a stream of charged particles emitted by the sun. When the solar wind reaches the Earth's magnetic field, it is deflected towards the poles, where it interacts with the atmosphere. The interaction between the solar wind and the atmosphere causes the atoms and molecules in the atmosphere to become excited, which leads to the emission of light. The col

## 🚀 Bölüm 9: HuggingFace'e Yükleme / Push to HuggingFace Hub



**Ne yükleniyor? / What's uploaded?**
- Sadece **LoRA adapter** (~100-200 MB) — tüm modeli değil
- Base model bilgisi (hangi modelden fine-tune edildiği)
- Tokenizer



In [14]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')  # yeni write token
login(token=hf_token)

In [15]:
# ⚠️ Kendi HuggingFace username'ini yaz!
# ⚠️ Write your own HuggingFace username!
HF_USERNAME  = 'Mervecaliskan'           # ← DEĞİŞTİR / CHANGE THIS
REPO_NAME    = f'{HF_USERNAME}/llama3.2-3b-dolly-qlora'

print(f'Yükleniyor / Uploading to: {REPO_NAME}')
print('Bu birkaç dakika sürebilir / This may take a few minutes...')

# LoRA adapter'ı yükle (tüm model değil — çok daha küçük)
# Push LoRA adapter only (not full model — much smaller)
model.push_to_hub(REPO_NAME, token=hf_token)
tokenizer.push_to_hub(REPO_NAME, token=hf_token)

# MLflow'a son bilgileri logla ve kapat
# Log final info to MLflow and close
mlflow.log_param('huggingface_repo', REPO_NAME)
mlflow.end_run()

print(f'\n✅ Model yüklendi / Model uploaded!')
print(f'🔗 https://huggingface.co/{REPO_NAME}')
print('📊 MLflow run kapatıldı / MLflow run closed')

Yükleniyor / Uploading to: Mervecaliskan/llama3.2-3b-dolly-qlora
Bu birkaç dakika sürebilir / This may take a few minutes...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 45.8kB / 97.3MB            

Saved model to https://huggingface.co/Mervecaliskan/llama3.2-3b-dolly-qlora


README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpuap016p1/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            


✅ Model yüklendi / Model uploaded!
🔗 https://huggingface.co/Mervecaliskan/llama3.2-3b-dolly-qlora
📊 MLflow run kapatıldı / MLflow run closed
